# Notebook 7 — Análisis Cualitativo Text-to-Outfit (RE Zero-shot · RE Fine-tuned · CLIP4Outfit)

Comparación directa: **la misma consulta de texto** evaluada por los tres modelos, con los outfits recuperados en el top-K (con imágenes) y la posición del outfit correcto en el ranking si se indica.

**Estructura:**
0. Setup
1. Carga de los tres modelos (checkpoint + recursos, calcado de sus notebooks de origen)
2. Función de comparación conjunta (misma consulta, los 3 modelos, con imágenes)
3. Casos de acierto (para 8.3.2)
4. Casos de fallo / consultas complejas (para 8.3.3 — Robustez ante Consultas Complejas)

## 0. Setup

In [ ]:
import json, re
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"
IMAGES_DIR = POLYVORE_PATH + "images/"

plt.rcParams.update({
    "font.family": "serif", "font.size": 11, "axes.titlesize": 12,
    "figure.dpi": 150,
})
MODEL_COLORS = {"RE Zero-shot": "tab:blue", "RE Fine-tuned": "tab:orange", "CLIP4Outfit": "tab:green"}

print(f"Device: {DEVICE}")


In [ ]:
# --- Mapeo set_id -> lista de item_ids (compartido por los 3 modelos, no depende del modelo) ---
def build_outfit_item_map(json_paths_with_split):
    outfit_items, outfit_split = {}, {}
    for path, split in json_paths_with_split:
        with open(path) as f:
            outfits = json.load(f)
        for outfit in outfits:
            set_id = str(outfit["set_id"])
            items_sorted = sorted(outfit["items"], key=lambda it: it["index"])
            outfit_items[set_id] = [str(it["item_id"]) for it in items_sorted]
            outfit_split[set_id] = split
    return outfit_items, outfit_split

OUTFIT_SPLIT_PATHS = [
    (POLYVORE_PATH + "nondisjoint/train.json", "train"),
    (POLYVORE_PATH + "nondisjoint/valid.json", "valid"),
    (POLYVORE_PATH + "nondisjoint/test.json",  "test"),
]
outfit_items, outfit_split = build_outfit_item_map(OUTFIT_SPLIT_PATHS)
print(f"Outfits indexados (set_id -> items): {len(outfit_items):,}")


def _load_item_img(item_id):
    try:
        return Image.open(IMAGES_DIR + f"{item_id}.jpg").convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), color="gray")


def plot_outfit_row(ax_row, set_id, max_items=6):
    """Pinta hasta max_items prendas de un outfit en una fila de ejes ya creada."""
    items = outfit_items.get(set_id, [])[:max_items]
    for ax in ax_row:
        ax.axis("off")
    for ax, item_id in zip(ax_row, items):
        ax.imshow(_load_item_img(item_id))
        ax.axis("off")


## 1. Carga de los tres modelos

### 1.1 Utilidades de texto (GloVe) -- compartidas por RE Zero-shot y RE Fine-tuned

In [ ]:
# Calcado de ambos notebooks RE (celdas 3, 5, 7) -- idénticos en los dos, así que se define una vez
GLOVE_PATH = POLYVORE_PATH + "glove/wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"
EMBED_DIM = 100

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Cargando GloVe (puede tardar un poco)...")
glove = {}
with open(GLOVE_PATH, encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        glove[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
print(f"GloVe cargado: {len(glove):,} tokens")

def text_to_glove_vector(text, glove=glove):
    tokens = clean_text(text).split()
    vecs = [glove[t] for t in tokens if t in glove]
    if not vecs:
        return np.zeros(EMBED_DIM, dtype=np.float32), True
    vec = np.mean(vecs, axis=0)
    vec = vec / (np.linalg.norm(vec) + 1e-8)
    return vec.astype(np.float32), False


### 1.2 Recommendation Engine — Text2Outfit Zero-shot

In [ ]:
# Calcado de Text-to-Outfit_Re_Zero-shot_02___multi-positive__.ipynb (celdas 9, 13, 15, 21, 23)
ZS_T2O_CACHE_PATH = POLYVORE_PATH + "checkpoints_RE/t2o_samples_with_embeddings.pt"
ZS_CKPT_PATH = POLYVORE_PATH + "checkpoints_RE/best_query_encoder_zero_shot.pt"

zs_samples = torch.load(ZS_T2O_CACHE_PATH, map_location=DEVICE)
for s in zs_samples:
    vec, _ = text_to_glove_vector(s["text"])
    s["query_glove_vec"] = torch.from_numpy(vec)
zs_samples = [s for s in zs_samples if not torch.all(s["query_glove_vec"] == 0)]
zs_test = [s for s in zs_samples if s["split"] == "test"]
print(f"[RE-ZS] muestras de test utilizables: {len(zs_test)}")

class QueryEncoderZS(nn.Module):
    def __init__(self, input_dim=EMBED_DIM, hidden_dim=256, output_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, glove_vec):
        return F.normalize(self.net(glove_vec), p=2, dim=-1)

zs_query_encoder = QueryEncoderZS().to(DEVICE)
zs_ckpt = torch.load(ZS_CKPT_PATH, map_location=DEVICE)
zs_query_encoder.load_state_dict(zs_ckpt["query_encoder_state_dict"])
zs_query_encoder.eval()
print(f"[RE-ZS] cargado -- época {zs_ckpt['epoch']}")

@torch.no_grad()
def zs_retrieve(query_text, candidates=zs_test, top_k=5):
    vec, is_oov = text_to_glove_vector(query_text)
    if is_oov:
        print("[RE-ZS] aviso: consulta totalmente OOV para GloVe.")
    q_emb = zs_query_encoder(torch.from_numpy(vec).unsqueeze(0).to(DEVICE))
    outfit_embs = torch.stack([s["outfit_emb"] for s in candidates]).to(DEVICE)
    sims = (q_emb @ outfit_embs.T).squeeze(0)
    order = sims.argsort(descending=True)
    ranked = [(candidates[i]["set_id"], sims[i].item()) for i in order.tolist()]
    return ranked   # lista completa ordenada -- top_k y rank del GT se extraen en la Sección 2


### 1.3 Recommendation Engine — Text2Outfit Fine-tuned

In [ ]:
# Calcado de Text-to-Outfit_RE_Fine-tuned_02.ipynb (celdas 9, 13, 17, 23)
FT_T2O_CACHE_PATH = POLYVORE_PATH + "checkpoints_RE/t2o_samples_with_embeddings_filtered_solocapauno.pt"
FT_CKPT_PATH = POLYVORE_PATH + "checkpoints_RE/best_query_encoder_finetuned_v3.pt"
D_MODEL, N_HEADS, FFN_DIM, DROPOUT = 128, 4, 256, 0.1

ft_cache = torch.load(FT_T2O_CACHE_PATH, map_location=DEVICE)
ft_test = ft_cache["test"]
for s in ft_test:
    vec, _ = text_to_glove_vector(s["text"])
    s["query_glove_vec"] = torch.from_numpy(vec)
ft_test = [s for s in ft_test if not torch.all(s["query_glove_vec"] == 0)]
print(f"[RE-FT] muestras de test utilizables: {len(ft_test)}")

class TransformerEncoderBlockFT(nn.Module):
    """Misma definicion que en el Recommendation Engine -- necesaria para cargar los pesos
    de la ultima capa desde el checkpoint original."""
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        return self.norm2(x + self.drop2(self.ffn(x)))

class QueryEncoderFT(nn.Module):
    def __init__(self, input_dim=EMBED_DIM, hidden_dim=256, output_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, glove_vec):
        return F.normalize(self.net(glove_vec), p=2, dim=-1)

ft_last_layer = TransformerEncoderBlockFT(D_MODEL, N_HEADS, FFN_DIM, DROPOUT).to(DEVICE)
ft_query_encoder = QueryEncoderFT().to(DEVICE)

ft_ckpt = torch.load(FT_CKPT_PATH, map_location=DEVICE)
ft_query_encoder.load_state_dict(ft_ckpt["query_encoder_state_dict"])
ft_last_layer.load_state_dict(ft_ckpt["last_layer_state_dict"])
ft_query_encoder.eval(); ft_last_layer.eval()
print(f"[RE-FT] cargado -- época {ft_ckpt['epoch']}")

def ft_compute_outfit_emb(prelast_seq, key_padding_mask):
    x = ft_last_layer(prelast_seq, key_padding_mask=key_padding_mask)
    return F.normalize(x[:, 0, :], p=2, dim=-1)

@torch.no_grad()
def ft_retrieve(query_text, candidates=ft_test, top_k=5):
    vec, is_oov = text_to_glove_vector(query_text)
    if is_oov:
        print("[RE-FT] aviso: consulta totalmente OOV para GloVe.")
    q_emb = ft_query_encoder(torch.from_numpy(vec).unsqueeze(0).to(DEVICE))
    prelast = torch.stack([s["prelast_seq"] for s in candidates]).to(DEVICE)
    mask = torch.stack([s["key_padding_mask"] for s in candidates]).to(DEVICE)
    outfit_embs = ft_compute_outfit_emb(prelast, mask)
    sims = (q_emb @ outfit_embs.T).squeeze(0)
    order = sims.argsort(descending=True)
    ranked = [(candidates[i]["set_id"], sims[i].item()) for i in order.tolist()]
    return ranked


### 1.4 CLIP4Outfit

In [ ]:
# Calcado de Clip4Outfit_03_..._gradcache_.ipynb (celdas 2, 4, 6, 8, 10, 12, 22)
from transformers import CLIPModel, CLIPTokenizerFast
from torchvision import transforms as tv_transforms

C4O_OUTFIT_TITLES_PATH = POLYVORE_PATH + "polyvore_outfit_titles.json"
C4O_CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
C4O_CLIP_FINETUNED_CKPT = POLYVORE_PATH + "checkpoints_clip4outfit/clip_finetuned_polyvore.pt"
C4O_CKPT_PATH = POLYVORE_PATH + "checkpoints_clip4outfit/clip4outfit_best_gradcache.pt"
C4O_IMG_SIZE = 224
C4O_MAX_TEXT_TOKENS = 24
C4O_D_MODEL = 512
C4O_MAX_ITEMS_PER_OUTFIT = 19

# --- Arquitectura (idéntica a la celda 4 de su notebook -- ver esa celda para el detalle
# completo de wti_similarity_batch / StyleTransformer / OutfitLevelEncoder) ---
P_WEIGHT, ALPHA_STYLE, BETA_OUTFIT, KO_RATIO, KT_RATIO = 0.2, 0.3, 0.3, 1/3, 1/6

def pad_and_mask(tensor_list):
    B = len(tensor_list); D = tensor_list[0].shape[-1]
    n_max = max(t.shape[0] for t in tensor_list)
    padded = torch.zeros(B, n_max, D, device=tensor_list[0].device, dtype=tensor_list[0].dtype)
    mask = torch.zeros(B, n_max, dtype=torch.bool, device=tensor_list[0].device)
    for i, t in enumerate(tensor_list):
        padded[i, :t.shape[0]] = t; mask[i, :t.shape[0]] = True
    return padded, mask

def pad_and_mask_images(image_list):
    B = len(image_list); C, H, W = image_list[0].shape[1:]
    n_max = max(t.shape[0] for t in image_list)
    padded = torch.zeros(B, n_max, C, H, W, device=image_list[0].device)
    mask = torch.zeros(B, n_max, dtype=torch.bool, device=image_list[0].device)
    for i, t in enumerate(image_list):
        padded[i, :t.shape[0]] = t; mask[i, :t.shape[0]] = True
    return padded, mask

def masked_softmax(logits, mask, dim=-1):
    logits = logits.masked_fill(~mask, float("-inf"))
    return torch.nan_to_num(F.softmax(logits, dim=dim), nan=0.0)

class AdaptiveWeightNet(nn.Module):
    def __init__(self, d_model):
        super().__init__(); self.linear = nn.Linear(d_model, 1)
    def forward(self, x, mask):
        return masked_softmax(self.linear(x).squeeze(-1), mask, dim=-1)

def wti_similarity_batch(item_emb, item_mask, text_emb, text_mask, fo_weight, ft_weight, p=P_WEIGHT):
    sim = torch.einsum("nid,mjd->nmij", item_emb, text_emb)
    joint_mask = item_mask.unsqueeze(1).unsqueeze(-1) & text_mask.unsqueeze(0).unsqueeze(2)
    sim = sim.masked_fill(~joint_mask, float("-inf"))
    m1 = torch.nan_to_num(sim.max(dim=3).values, neginf=0.0)
    term1 = (fo_weight.unsqueeze(1) * m1).sum(dim=2)
    m2 = torch.nan_to_num(sim.max(dim=2).values, neginf=0.0)
    term2 = (ft_weight.unsqueeze(0) * m2).sum(dim=2)
    return (term1 + p * term2) / (1 + p)

class StyleTransformer(nn.Module):
    def __init__(self, d_model, n_heads=8, ffn_dim=None, dropout=0.1):
        super().__init__()
        ffn_dim = ffn_dim or d_model * 4
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(ffn_dim, d_model))
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout); self.drop2 = nn.Dropout(dropout)
    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        return self.norm2(x + self.drop2(self.ffn(x)))

@torch.no_grad()
def greedy_kmeans_init(X, k):
    dists = torch.cdist(X, X)
    chosen = [dists.sum(dim=1).argmax().item()]
    for _ in range(1, k):
        d_to_centroids = torch.cdist(X, X[chosen])
        min_d = d_to_centroids.min(dim=1).values
        min_d[chosen] = -float("inf")
        chosen.append(min_d.argmax().item())
    return X[chosen].clone()

@torch.no_grad()
def kmeans_assign(X, k, n_iters=5):
    centroids = greedy_kmeans_init(X, k)
    assign = torch.zeros(X.shape[0], dtype=torch.long, device=X.device)
    for _ in range(n_iters):
        assign = torch.cdist(X, centroids).argmin(dim=1)
        for c in range(k):
            members = X[assign == c]
            if members.shape[0] > 0:
                centroids[c] = members.mean(dim=0)
    return assign

def style_extractor(X, k_ratio):
    N = X.shape[0]
    if N <= 1:
        return X
    k = min(max(1, int(k_ratio * N)), N)
    assign = kmeans_assign(X.detach(), k)
    centroids = [X[assign == c].mean(dim=0) for c in range(k) if (assign == c).any()]
    if not centroids:
        centroids = [X.mean(dim=0)]
    return F.normalize(torch.stack(centroids), p=2, dim=-1)

class OutfitLevelEncoder(nn.Module):
    def __init__(self, d_model, n_heads=8, dropout=0.1):
        super().__init__(); self.transformer = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
    def forward(self, x, key_padding_mask=None):
        x = self.transformer(x, key_padding_mask=key_padding_mask)
        real_mask = (~key_padding_mask).unsqueeze(-1).float() if key_padding_mask is not None \
            else torch.ones(x.shape[0], x.shape[1], 1, device=x.device)
        pooled = (x * real_mask).sum(dim=1) / real_mask.sum(dim=1).clamp(min=1.0)
        return F.normalize(pooled, p=2, dim=-1)

class CLIP4OutfitCore(nn.Module):
    def __init__(self, d_model=C4O_D_MODEL, n_heads=8, dropout=0.1,
                 ko_ratio=KO_RATIO, kt_ratio=KT_RATIO, p_weight=P_WEIGHT):
        super().__init__()
        self.ko_ratio, self.kt_ratio, self.p_weight = ko_ratio, kt_ratio, p_weight
        self.fo_item = AdaptiveWeightNet(d_model); self.ft_item = AdaptiveWeightNet(d_model)
        self.style_transformer_o = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
        self.style_transformer_t = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
        self.fo_style = AdaptiveWeightNet(d_model); self.ft_style = AdaptiveWeightNet(d_model)
        self.outfit_encoder_o = OutfitLevelEncoder(d_model, n_heads=n_heads, dropout=dropout)
        self.outfit_encoder_t = OutfitLevelEncoder(d_model, n_heads=n_heads, dropout=dropout)

    def forward(self, item_emb, item_mask, text_emb, text_mask):
        B = item_emb.shape[0]
        fo_i = self.fo_item(item_emb, item_mask); ft_i = self.ft_item(text_emb, text_mask)
        S_item = wti_similarity_batch(item_emb, item_mask, text_emb, text_mask, fo_i, ft_i, p=self.p_weight)

        e_prime_o = self.style_transformer_o(item_emb, key_padding_mask=~item_mask)
        e_prime_t = self.style_transformer_t(text_emb, key_padding_mask=~text_mask)
        style_o_list = [style_extractor(e_prime_o[b][item_mask[b]], self.ko_ratio) for b in range(B)]
        style_t_list = [style_extractor(e_prime_t[b][text_mask[b]], self.kt_ratio) for b in range(text_emb.shape[0])]
        style_o, style_o_mask = pad_and_mask(style_o_list)
        style_t, style_t_mask = pad_and_mask(style_t_list)

        fo_s = self.fo_style(style_o, style_o_mask); ft_s = self.ft_style(style_t, style_t_mask)
        S_style = wti_similarity_batch(style_o, style_o_mask, style_t, style_t_mask, fo_s, ft_s, p=self.p_weight)

        o_vec = self.outfit_encoder_o(style_o, key_padding_mask=~style_o_mask)
        t_vec = self.outfit_encoder_t(style_t, key_padding_mask=~style_t_mask)
        S_outfit = o_vec @ t_vec.T
        return S_item, S_style, S_outfit

def extract_image_embeds(output):
    if torch.is_tensor(output):
        return output
    if getattr(output, "image_embeds", None) is not None:
        return output.image_embeds
    if getattr(output, "pooler_output", None) is not None:
        return output.pooler_output
    raise TypeError(f"No se pudo extraer embeddings de imagen de {type(output)}")

def c4o_encode_items(clip_model, item_images, item_mask):
    B, No_max = item_images.shape[:2]
    flat = item_images.reshape(B * No_max, 3, C4O_IMG_SIZE, C4O_IMG_SIZE)
    embeds = F.normalize(extract_image_embeds(clip_model.get_image_features(pixel_values=flat)), p=2, dim=-1)
    embeds = embeds.reshape(B, No_max, -1) * item_mask.unsqueeze(-1)
    return embeds

def c4o_encode_text(clip_model, input_ids, attention_mask):
    outputs = clip_model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    token_embeds = F.normalize(clip_model.text_projection(outputs.last_hidden_state), p=2, dim=-1)
    text_mask = attention_mask.bool()
    return token_embeds * text_mask.unsqueeze(-1), text_mask

# --- Carga de checkpoint ---
clip_model = CLIPModel.from_pretrained(C4O_CLIP_MODEL_NAME).to(DEVICE)
tokenizer = CLIPTokenizerFast.from_pretrained(C4O_CLIP_MODEL_NAME)
core_model = CLIP4OutfitCore().to(DEVICE)

c4o_ckpt = torch.load(C4O_CKPT_PATH, map_location=DEVICE)
clip_model.load_state_dict(c4o_ckpt["clip_state_dict"])
core_model.load_state_dict(c4o_ckpt["core_state_dict"])
clip_model.eval(); core_model.eval()
print(f"[CLIP4Outfit] cargado -- época {c4o_ckpt['epoch']}")

# --- Reconstruir t2o_test (CLIP4Outfit no cachea sus samples en disco, a diferencia de RE) ---
def clean_text_basic(s):
    return " ".join(s.strip().split())

def get_outfit_text(entry):
    title = clean_text_basic(entry.get("title", ""))
    return title if title else clean_text_basic(entry.get("url_name", ""))

STORE_PATTERN = re.compile(r"\b(yesstyle|banggood|romwe|shein|zaful|aliexpress|paoloshoes|dhgate|rosegal|wish)\b", re.I)
PROMO_PATTERN = re.compile(r"\b(off|coupon|code|sale|discount|deal|free shipping)\b", re.I)
SIZE_PATTERN  = re.compile(r"\b(plus size|size \d|us \d|eu \d)\b", re.I)

def is_spam_text(text):
    return bool(STORE_PATTERN.search(text) or PROMO_PATTERN.search(text) or SIZE_PATTERN.search(text))

import pandas as pd
df_valid_items = pd.read_csv(POLYVORE_PATH + "data/processed/glove_text_features.csv")
valid_item_ids_with_image = set(df_valid_items["item_id"].astype(str))

with open(C4O_OUTFIT_TITLES_PATH) as f:
    outfit_titles_raw = json.load(f)

c4o_samples = []
for set_id, entry in outfit_titles_raw.items():
    text = get_outfit_text(entry)
    if not text or is_spam_text(text):
        continue
    item_ids = outfit_items.get(set_id)
    if not item_ids or len(item_ids) > C4O_MAX_ITEMS_PER_OUTFIT:
        continue
    if not all(iid in valid_item_ids_with_image for iid in item_ids):
        continue
    c4o_samples.append({"set_id": set_id, "item_ids": item_ids, "text": text,
                         "split": outfit_split.get(set_id, "unknown")})

c4o_test = [s for s in c4o_samples if s["split"] == "test"]
print(f"[CLIP4Outfit] muestras de test reconstruidas: {len(c4o_test)}")

# --- Precomputar item_emb/item_mask de TODO el test una sola vez (caro -- imágenes reales) ---
clip_image_transform = tv_transforms.Compose([
    tv_transforms.Resize(C4O_IMG_SIZE, interpolation=tv_transforms.InterpolationMode.BICUBIC),
    tv_transforms.CenterCrop(C4O_IMG_SIZE), tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)),
])

def load_clip_image(item_id):
    img = Image.open(IMAGES_DIR + f"{item_id}.jpg").convert("RGB")
    return clip_image_transform(img)

@torch.no_grad()
def c4o_precompute_item_embeddings(samples, batch_size=16):
    all_embs, all_masks = [], []
    for i in tqdm(range(0, len(samples), batch_size), desc="[CLIP4Outfit] Precomputando item embeddings"):
        batch = samples[i:i + batch_size]
        images_list = [torch.stack([load_clip_image(iid) for iid in s["item_ids"]]) for s in batch]
        item_images, item_mask = pad_and_mask_images(images_list)
        item_images, item_mask = item_images.to(DEVICE), item_mask.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            item_emb = c4o_encode_items(clip_model, item_images, item_mask)
        all_embs.append(item_emb.cpu()); all_masks.append(item_mask.cpu())
    n_max = max(e.shape[1] for e in all_embs)
    def repad(t, m):
        if t.shape[1] < n_max:
            pad_t = torch.zeros(t.shape[0], n_max - t.shape[1], t.shape[2])
            pad_m = torch.zeros(m.shape[0], n_max - m.shape[1], dtype=torch.bool)
            t = torch.cat([t, pad_t], dim=1); m = torch.cat([m, pad_m], dim=1)
        return t, m
    # Una sola pasada por (embedding, mascara) ORIGINALES -- antes se llamaba repad() dos
    # veces por separado y la 2a ya recibia el embedding REPADEADO de la 1a, dejando la
    # mascara sin repadear (de ahi el mismatch de tamanos al concatenar).
    padded_pairs = [repad(e, m) for e, m in zip(all_embs, all_masks)]
    all_embs  = [p[0] for p in padded_pairs]
    all_masks = [p[1] for p in padded_pairs]
    return torch.cat(all_embs, dim=0), torch.cat(all_masks, dim=0)

from tqdm import tqdm
c4o_item_emb_all, c4o_item_mask_all = c4o_precompute_item_embeddings(c4o_test)
print(f"[CLIP4Outfit] item_emb precomputados: {tuple(c4o_item_emb_all.shape)}")

@torch.no_grad()
def c4o_retrieve(query_text, candidates=c4o_test, item_emb_all=None, item_mask_all=None, top_k=5):
    item_emb_all = c4o_item_emb_all if item_emb_all is None else item_emb_all
    item_mask_all = c4o_item_mask_all if item_mask_all is None else item_mask_all
    tok = tokenizer([query_text], padding="max_length", truncation=True,
                     max_length=C4O_MAX_TEXT_TOKENS, return_tensors="pt")
    text_emb, text_mask = c4o_encode_text(clip_model, tok["input_ids"].to(DEVICE), tok["attention_mask"].to(DEVICE))

    scores = []
    eval_batch = 64
    for i in range(0, item_emb_all.shape[0], eval_batch):
        ie = item_emb_all[i:i + eval_batch].to(DEVICE)
        im = item_mask_all[i:i + eval_batch].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            _, _, S_outfit = core_model(ie, im, text_emb, text_mask)
        scores.append(S_outfit[:, 0].float().cpu())
    scores = torch.cat(scores)
    order = scores.argsort(descending=True)
    ranked = [(candidates[i]["set_id"], scores[i].item()) for i in order.tolist()]
    return ranked


## 2. Función de comparación conjunta

Ejecuta la MISMA consulta en los 3 modelos y muestra, para cada uno, los `top_k` outfits recuperados (con imágenes) en una fila. Si se indica `correct_set_id`, se marca en verde si aparece dentro del top-k mostrado, y se imprime su posición exacta en el ranking completo (no solo si está o no en el top-k) -- incluso si no aparece en las filas mostradas.

In [ ]:
def rank_of(ranked, set_id):
    """Posición (1-indexada) de set_id en la lista `ranked` (ya ordenada desc.), o None si no está."""
    for i, (sid, _) in enumerate(ranked, start=1):
        if sid == set_id:
            return i
    return None


def compare_query(query_text, correct_set_id=None, top_k=5, max_items_per_outfit=6):
    """Corre la consulta en RE-ZS, RE-FT y CLIP4Outfit. Visualiza el OUTFIT COMPLETO (todas
    sus prendas, no una sola) del top-1 recuperado por cada modelo -- y el outfit correcto
    completo, si se pasa `correct_set_id`, en su propia fila para comparar directamente.
    El top-k con scores se imprime como texto (ahí sí interesan los k, no hace falta pintar
    cada uno como outfit completo -- saturaría la figura)."""
    print(f'Consulta: "{query_text}"')
    if correct_set_id is not None:
        print(f"Outfit correcto (GT): set_id={correct_set_id}")

    rankings = {
        "RE Zero-shot": zs_retrieve(query_text, top_k=top_k),
        "RE Fine-tuned": ft_retrieve(query_text, top_k=top_k),
        "CLIP4Outfit":   c4o_retrieve(query_text, top_k=top_k),
    }

    # --- Texto: ranking completo top-k por modelo ---
    for name, ranked in rankings.items():
        gt_rank = rank_of(ranked, correct_set_id) if correct_set_id is not None else None
        gt_txt = f"  (GT en posición {gt_rank})" if gt_rank else \
                 ("  (GT fuera del ranking mostrado)" if correct_set_id is not None else "")
        print(f"\n{name}{gt_txt}")
        for i, (set_id, score) in enumerate(ranked[:top_k], start=1):
            marker = " <- GT" if set_id == correct_set_id else ""
            print(f"   #{i}  s={score:.3f}  {set_id}{marker}")

    # --- Figura: outfit COMPLETO del top-1 de cada modelo (+ GT si se indica) ---
    n_rows = len(rankings) + (1 if correct_set_id is not None else 0)
    fig, axes = plt.subplots(n_rows, max_items_per_outfit, figsize=(2.2 * max_items_per_outfit, 2.4 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    row = 0
    for name, ranked in rankings.items():
        top1_set_id, top1_score = ranked[0]
        plot_outfit_row(axes[row], top1_set_id, max_items=max_items_per_outfit)
        is_correct = (top1_set_id == correct_set_id)
        color = "green" if is_correct else MODEL_COLORS.get(name, "black")
        label = f"{name} — Top-1 (s={top1_score:.3f})" + ("  ✓" if is_correct else "")
        axes[row, 0].annotate(label, xy=(-0.15, 0.5), xycoords="axes fraction",
                               fontsize=9.5, ha="right", va="center", weight="bold", color=color)
        row += 1

    if correct_set_id is not None:
        plot_outfit_row(axes[row], correct_set_id, max_items=max_items_per_outfit)
        axes[row, 0].annotate("Outfit correcto (GT)", xy=(-0.15, 0.5), xycoords="axes fraction",
                               fontsize=9.5, ha="right", va="center", weight="bold", color="green")

    plt.tight_layout()
    plt.show()

    result = {"query": query_text, "correct_set_id": correct_set_id, "rankings": {}}
    for name, ranked in rankings.items():
        result["rankings"][name] = {
            "top_k": ranked[:top_k],
            "gt_rank": rank_of(ranked, correct_set_id) if correct_set_id is not None else None,
        }
    return result


## 3. Casos de acierto (para 8.3.2)

Para que la comparación sea justa, la consulta debe existir en el conjunto de test de los 3 modelos a la vez (cada uno reconstruye/filtra su propio `test` de forma algo distinta -- `zs_test`, `ft_test`, `c4o_test`). `find_common_queries` localiza esos casos comunes; a partir de ahí, ejecuta `compare_query` sobre los que consideres más ilustrativos como acierto (idealmente donde el GT quede en el top-k para los 3, o al menos para 2 de los 3 -- así se ve el contraste).

In [ ]:
import random

In [ ]:
def find_common_queries(n=15, seed=0, min_text_len=15):
    zs_by_text = {s["text"]: s["set_id"] for s in zs_test}
    ft_by_text = {s["text"]: s["set_id"] for s in ft_test}
    c4o_by_text = {s["text"]: s["set_id"] for s in c4o_test}
    common_texts = [t for t in zs_by_text if t in ft_by_text and t in c4o_by_text and len(t) >= min_text_len]
    print(f"Consultas de texto presentes en los 3 test sets a la vez: {len(common_texts)}")

    rng = random.Random(seed)
    sample_texts = rng.sample(common_texts, min(n, len(common_texts)))
    # OJO: el set_id "correcto" es el mismo texto -> mismo outfit en los 3, porque viene
    # del mismo polyvore_outfit_titles.json -- si algún día difiere, es señal de un problema
    # de construcción de alguno de los 3 t2o_* upstream, no de esta celda.
    return [(t, zs_by_text[t]) for t in sample_texts]

candidate_queries = find_common_queries(n=15)
for text, set_id in candidate_queries:
    print(f"  {set_id:>10}  {text}")


In [ ]:
# Ejecuta compare_query sobre las consultas candidatas -- revisa cuáles quedan bien como
# ACIERTO (GT en el top-k, sobre todo para los 3 modelos o para 2/3) y guarda esas para 8.3.2.
success_examples = []
for text, set_id in candidate_queries[:5]:
    result = compare_query(text, correct_set_id=set_id, top_k=5)
    success_examples.append(result)


## 4. Casos de fallo / consultas complejas (para 8.3.3 — Robustez ante Consultas Complejas)

Dos fuentes de casos distintas:
1. **Fallos reales** entre `candidate_queries` de la Sección 3 -- las que salieron con GT fuera del top-k mostrado en alguno de los 3 modelos.
2. **Consultas complejas construidas a propósito** (compuestas, con negación, con atributos poco frecuentes) -- sin `correct_set_id` necesariamente, para juzgar solo cualitativamente qué recupera cada modelo cuando la consulta se aleja del tipo de frase típico de `polyvore_outfit_titles.json`.

In [ ]:
# 4.1 -- fallos reales encontrados en la Seccion 3 (ya ejecutados, no hay que rehacer las consultas)
failure_examples = [
    r for r in success_examples
    if any(v["gt_rank"] is None or v["gt_rank"] > 5 for v in r["rankings"].values())
]
print(f"De los {len(success_examples)} ejemplos de la Sección 3, "
      f"{len(failure_examples)} tienen el GT fuera del top-5 en al menos un modelo.")
for r in failure_examples:
    ranks_txt = ", ".join(f"{name}: rank={v['gt_rank']}" for name, v in r["rankings"].items())
    print(f'  "{r["query"]}"  ->  {ranks_txt}')


In [ ]:
# 4.2 -- consultas complejas construidas a propósito (AJUSTAR a lo que tenga sentido para tu
# dataset/discusión -- estas son solo ejemplos de partida, mezclando: consulta compuesta,
# negación, atributo poco frecuente, y consulta muy larga/detallada)
complex_queries = [
    "casual outfit for a rainy day but not too warm",
    "elegant evening look without black or heels",
    "streetwear inspired by 90s hip hop with oversized silhouette",
    "minimalist monochrome outfit with exactly one statement accessory",
]

complex_results = []
for q in complex_queries:
    complex_results.append(compare_query(q, correct_set_id=None, top_k=5))


## 5. Parámetros y tiempo de inferencia

**Aviso metodológico, igual que en el notebook comparativo de las otras 3 arquitecturas: no es una comparación simétrica en lo que cada modelo computa por consulta**, y hay que decirlo explícitamente en la memoria en vez de dejar que la tabla hable sola:

- **RE Zero-shot**: los embeddings de outfit están **precalculados y cacheados** (`s["outfit_emb"]`) -- por consulta solo se calcula el embedding del texto y un producto escalar contra el caché. Es la opción más barata en inferencia.
- **RE Fine-tuned**: `prelast_seq` está cacheado, pero `last_layer` (la única capa reentrenada) se **recalcula en cada consulta** para todos los candidatos -- más caro que Zero-shot aunque comparta casi todo el pipeline.
- **CLIP4Outfit**: los embeddings de imagen de los candidatos están precalculados (Sección 1.4), pero por consulta hay que pasar el texto por el **encoder de texto completo de CLIP** y evaluar `core_model` (con su mecanismo de atención + extracción de estilo) contra todos los candidatos -- el más caro de los tres, y además el modelo con más parámetros con diferencia (CLIP ViT-B/32 completo + `core_model`).

En parámetros, comparar el nº total de CLIP4Outfit (que incluye el backbone visual+texto completo de CLIP) contra el de un `query_encoder` de unas pocas capas MLP es comparar cosas de una escala completamente distinta -- soporta la discusión de coste/beneficio, no una afirmación de "arquitectura más eficiente".

In [ ]:
def n_params(model) -> int:
    return sum(p.numel() for p in model.parameters())

def time_inference(fn, *args, n_reps=30, n_warmup=5, **kwargs):
    for _ in range(n_warmup):
        fn(*args, **kwargs)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return {"mean_ms": float(np.mean(times)), "std_ms": float(np.std(times)), "n_reps": n_reps}

import time


### 5.1 Parámetros

In [ ]:
params_table = {
    "RE Zero-shot":  n_params(zs_query_encoder),
    "RE Fine-tuned": n_params(ft_query_encoder) + n_params(ft_last_layer),
    "CLIP4Outfit":   n_params(clip_model) + n_params(core_model),
}
for name, n in params_table.items():
    print(f"{name:<15} {n:>14,} parámetros")

print(f"\n(Solo referencia: CLIP4Outfit desglosado -- CLIP: {n_params(clip_model):,}  "
      f"core_model: {n_params(core_model):,})")


### 5.2 Tiempo de inferencia por consulta

In [ ]:
_timing_query = "sweater dresses"   # misma consulta fija para los 3, real y ya usada arriba

def _zs_call():
    zs_retrieve(_timing_query, top_k=5)

def _ft_call():
    ft_retrieve(_timing_query, top_k=5)

def _c4o_call():
    c4o_retrieve(_timing_query, top_k=5)

timing_table = {}
for name, fn in [("RE Zero-shot", _zs_call), ("RE Fine-tuned", _ft_call), ("CLIP4Outfit", _c4o_call)]:
    t = time_inference(fn, n_reps=20, n_warmup=3)
    timing_table[name] = t
    print(f"{name:<15} {t['mean_ms']:8.2f} \u00b1 {t['std_ms']:6.2f} ms  "
          f"(sobre {len(zs_test) if name=='RE Zero-shot' else (len(ft_test) if name=='RE Fine-tuned' else len(c4o_test))} candidatos)")
